In [ ]:
from PIL import Image, ImageOps
import numpy as np

def load_img_consistent(path, max_size=1200):
    img = Image.open(path)
    img = ImageOps.exif_transpose(img)
    
    w, h = img.size
    scale = max(w, h) / max_size
    print(scale)
    if scale > 1:
        new_size = (int(w / scale), int(h / scale))
        img = img.resize(new_size, Image.LANCZOS)
        print(f"Resized from {w}×{h} → {new_size}")
    
    return np.array(img)

def normalize_points(pts):
    mean = pts.mean(axis=0)
    std  = pts.std(axis=0).mean()
    s = np.sqrt(2) / std
    T = np.array([[s, 0, -s*mean[0]],
                  [0, s, -s*mean[1]],
                  [0, 0, 1]])
    pts_h = np.c_[pts, np.ones(len(pts))].T
    npts = (T @ pts_h).T[:, :2]
    return npts, T

import numpy as np

def reprojection_errors(H, im1_pts, im2_pts):
    #im1 -> im2
    n = im1_pts.shape[0]
    p1 = np.c_[im1_pts, np.ones(n)]
    p2_hat = (H @ p1.T).T
    p2_hat = p2_hat[:, :2] / p2_hat[:, 2:3]

    #im2 -> im1 (consistency)
    Hinv = np.linalg.inv(H)
    p2 = np.c_[im2_pts, np.ones(n)]
    p1_hat = (Hinv @ p2.T).T
    p1_hat = p1_hat[:, :2] / p1_hat[:, 2:3]

    err_fwd = np.linalg.norm(p2_hat - im2_pts, axis=1)
    err_bwd = np.linalg.norm(p1_hat - im1_pts, axis=1)
    return err_fwd, err_bwd

def warp_bounds(im, H):
    h, w = im.shape[:2]
    corners = np.array([[0,0,1],[w,0,1],[0,h,1],[w,h,1]]).T
    wc = H @ corners
    wc /= wc[2]
    xs, ys = wc[0], wc[1]
    min_x, max_x = np.floor(xs.min()), np.ceil(xs.max())
    min_y, max_y = np.floor(ys.min()), np.ceil(ys.max())
    tx = -min_x if min_x < 0 else 0
    ty = -min_y if min_y < 0 else 0
    T = np.array([[1,0,tx],[0,1,ty],[0,0,1]])
    out_w = int(max_x - min_x)
    out_h = int(max_y - min_y)
    return (out_h, out_w), T



In [ ]:
def computeH(im1_pts, im2_pts):
    # nnormalize
    p1n, T1 = normalize_points(im1_pts)
    p2n, T2 = normalize_points(im2_pts)

    N = im1_pts.shape[0]
    A = []
    for i in range(N):
        x, y = p1n[i]
        u, v = p2n[i]
        A.append([ x, y, 1, 0, 0, 0, -u*x, -u*y, -u])
        A.append([ 0, 0, 0, x, y, 1, -v*x, -v*y, -v])
    A = np.array(A)

    _, _, Vt = np.linalg.svd(A)
    Hn = Vt[-1].reshape(3,3)
    # Denormalize:
    # # H = T2^{-1} Hn T1
    H = np.linalg.inv(T2) @ Hn @ T1
    return H / H[2,2]

In [22]:
%matplotlib qt

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np

def pick_points(image_path, n, title):
    img = load_img_consistent(image_path)
    fig, ax = plt.subplots()
    ax.imshow(img)  
    # no rotation now
    ax.set_title(title)
    pts = np.array(plt.ginput(n, timeout=-1, show_clicks=True), dtype=float)
    plt.close(fig)
    return pts


N = 6  # 
im1_path = "/Users/jaitegchahal/cs180/jaitegchahal123.github.io/proj3/media/IMG_5781.jpg"
im2_path = "/Users/jaitegchahal/cs180/jaitegchahal123.github.io/proj3/media/IMG_5782.jpg"

print(f"Select {N} points on LEFT image (in order).")
im1_pts = pick_points(im1_path, N, "LEFT image")

print(f"Select {N} MATCHING points on RIGHT image in the SAME order.")
im2_pts = pick_points(im2_path, N, "RIGHT image")

# im1_pts[i] ↔ im2_pts[i]


Select 6 points on LEFT image (in order).
4.76
Resized from 4284×5712 → (900, 1200)
Select 6 MATCHING points on RIGHT image in the SAME order.
4.76
Resized from 4284×5712 → (900, 1200)


In [28]:
print(im1_pts)
print(im2_pts)

[[683.26623377 396.9025974 ]
 [680.01948052 546.25324675]
 [673.52597403 689.11038961]
 [868.33116883 374.17532468]
 [855.34415584 640.40909091]
 [842.35714286 864.43506494]]
[[147.55194805 348.2012987 ]
 [157.29220779 504.04545455]
 [160.53896104 653.3961039 ]
 [322.87662338 357.94155844]
 [326.12337662 594.95454545]
 [322.87662338 796.25324675]]


In [29]:
H = computeH(im1_pts, im2_pts)
print(H)

[[ 3.54844621e+00  2.24214428e-01 -2.08217650e+03]
 [ 1.46284377e+00  3.12993924e+00 -1.24166168e+03]
 [ 2.62669334e-03  2.04379444e-04  1.00000000e+00]]


In [ ]:
import matplotlib.pyplot as plt
im1 = load_img_consistent(im1_path)
im2 = load_img_consistent(im2_path)
def visualize_projection(H, im1, im2, im1_pts, im2_pts, title=""):
    # project im1_pts -> im2 frame
    p1 = np.c_[im1_pts, np.ones(len(im1_pts))]
    proj = (H @ p1.T).T
    proj = proj[:, :2] / proj[:, 2:3]

    plt.figure(figsize=(7,7))
    plt.imshow(im2)
    plt.scatter(im2_pts[:,0], im2_pts[:,1], s=40, c='lime', label='clicked im2_pts')
    plt.scatter(proj[:,0],   proj[:,1],   s=40, facecolors='none', edgecolors='red', label='H * im1_pts')
    for i,(a,b) in enumerate(zip(im2_pts, proj)):
        plt.plot([a[0], b[0]], [a[1], b[1]], 'y-', linewidth=1)
        plt.text(a[0]+3, a[1]+3, str(i), color='white', fontsize=9)
        plt.text(b[0]+3, b[1]+3, str(i), color='red', fontsize=9)
    plt.legend(); plt.title(title); plt.axis('off'); plt.show()

visualize_projection(H, im1, im2, im1_pts, im2_pts, "Projection check: im1 -> im2")


4.76
Resized from 4284×5712 → (900, 1200)
4.76
Resized from 4284×5712 → (900, 1200)


2025-10-08 20:55:01.655 python[39037:4831001] +[CATransaction synchronize] called within transaction
2025-10-08 20:55:02.125 python[39037:4831001] +[CATransaction synchronize] called within transaction
2025-10-08 20:55:23.816 python[39037:4831001] +[CATransaction synchronize] called within transaction
2025-10-08 20:55:25.379 python[39037:4831001] +[CATransaction synchronize] called within transaction


In [31]:
def warpImageNearestNeighbor(im, H, out_shape):
    H_inv = np.linalg.inv(H)
    h_out, w_out = out_shape
    warped_im = np.zeros((h_out, w_out, im.shape[2]), dtype=im.dtype)

    for i in range(h_out):
        for j in range(w_out):
            p_out = np.array([j, i, 1])
            p_in = H_inv @ p_out
            p_in /= p_in[2]
            x_in, y_in = int(round(p_in[0])), int(round(p_in[1]))

            if 0 <= x_in < im.shape[1] and 0 <= y_in < im.shape[0]:
                warped_im[i, j] = im[y_in, x_in]

    return warped_im

In [32]:
def warpImageBilinear(im, H, out_shape):
    H_inv = np.linalg.inv(H)
    h_out, w_out = out_shape
    warped_im = np.zeros((h_out, w_out, im.shape[2]), dtype=im.dtype)

    for i in range(h_out):
        for j in range(w_out):
            p_out = np.array([j, i, 1])
            p_in = H_inv @ p_out
            p_in /= p_in[2]
            x_in, y_in = p_in[0], p_in[1]

            if 0 <= x_in < im.shape[1]-1 and 0 <= y_in < im.shape[0]-1:
                x0, y0 = int(np.floor(x_in)), int(np.floor(y_in))
                x1, y1 = x0 + 1, y0 + 1

                dx, dy = x_in - x0, y_in - y0

                for c in range(im.shape[2]):
                    top = (1 - dx) * im[y0, x0, c] + dx * im[y0, x1, c]
                    bottom = (1 - dx) * im[y1, x0, c] + dx * im[y1, x1, c]
                    warped_im[i, j, c] = (1 - dy) * top + dy * bottom

    return warped_im


In [ ]:
im1 = load_img_consistent(im1_path)
im2 = load_img_consistent(im2_path)
h_out, w_out = im2.shape[0], im2.shape[1]  
#canvas = im2’s frame

imwarped_nn = warpImageNearestNeighbor(im1, H, (h_out, w_out))
imwarped_bilinear = warpImageBilinear(im1, H, (h_out, w_out))

fig, axs = plt.subplots(1, 2, figsize=(10, 5))
axs[0].imshow(imwarped_nn)
axs[0].set_title("Nearest Neighbor")
axs[1].imshow(imwarped_bilinear)
axs[1].set_title("Bilinear Interpolation")
plt.show()

4.76
Resized from 4284×5712 → (900, 1200)
4.76
Resized from 4284×5712 → (900, 1200)


In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.imshow(imwarped_nn)
plt.title("Nearest Neighbor")
plt.axis('off')

rando = Image.fromarray(imwarped_nn)
rando.save("./media/warpednn.png")

plt.figure()
plt.imshow(imwarped_bilinear)
plt.title("Bilinear Interpolation")
plt.axis('off')

rand1 = Image.fromarray(imwarped_bilinear)
rand1.save("./media/warpedbil.png")

plt.show()


In [1]:
#Rectification
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# 1️⃣ Load one tilted image
im_path = "/Users/jaitegchahal/cs180/jaitegchahal123.github.io/proj3/media/IMG_5868.jpg"
im = load_img_consistent(im_path)

# 2️⃣ Select 4 points on the tilted rectangle in the image (clockwise order)
print("Select 4 corners of a rectangular object (in order).")
im_pts = pick_points(im_path, 4, "Select 4 corners of the tilted rectangle")

# 3️⃣ Define target points as a perfect rectangle (e.g., 600x400 pixels)
w, h = 600, 400
rect_pts = np.array([[0, 0], [w, 0], [w, h], [0, h]], dtype=float)

# 4️⃣ Compute homography (maps image → rectified rectangle)
H_rect = computeH(im_pts, rect_pts)

# 5️⃣ Warp to rectangular view
rectified_nn = warpImageNearestNeighbor(im, H_rect, (h, w))
rectified_bil = warpImageBilinear(im, H_rect, (h, w))

# 6️⃣ Show results
fig, axs = plt.subplots(1, 2, figsize=(10, 5))
axs[0].imshow(rectified_nn)
axs[0].set_title("Rectified (Nearest Neighbor)")
axs[1].imshow(rectified_bil)
axs[1].set_title("Rectified (Bilinear Interpolation)")
plt.show()

# 7️⃣ (optional) Save the rectified image
Image.fromarray(rectified_bil.astype(np.uint8)).save("./media/rectified.jpg")


NameError: name 'load_img_consistent' is not defined

In [35]:
def mosaic_bounds(im1, im2, H):
    h1, w1 = im1.shape[:2]
    h2, w2 = im2.shape[:2]
    corners1 = np.array([[0,0,1],[w1,0,1],[0,h1,1],[w1,h1,1]]).T
    corners2 = np.array([[0,0,1],[w2,0,1],[0,h2,1],[w2,h2,1]]).T

    warped_corners1 = H @ corners1
    warped_corners1 /= warped_corners1[2]

    all_x = np.hstack([warped_corners1[0], corners2[0]])
    all_y = np.hstack([warped_corners1[1], corners2[1]])

    min_x, max_x = np.floor(all_x.min()), np.ceil(all_x.max())
    min_y, max_y = np.floor(all_y.min()), np.ceil(all_y.max())

    tx = -min_x if min_x < 0 else 0
    ty = -min_y if min_y < 0 else 0
    T = np.array([[1,0,tx],[0,1,ty],[0,0,1]])
    out_w, out_h = int(max_x - min_x), int(max_y - min_y)
    return (out_h, out_w), T


In [36]:
out_shape, T = mosaic_bounds(im1, im2, H)
H_total = T @ H
im1_warped = warpImageBilinear(im1, H_total, out_shape)


In [ ]:
mosaic = np.zeros_like(im1_warped)
h2, w2 = im2.shape[:2]

for i in range(h2):
    for j in range(w2):
        x, y = int(j + T[0,2]), int(i + T[1,2])
        if 0 <= y < mosaic.shape[0] and 0 <= x < mosaic.shape[1]:
            mosaic[y, x] = im2[i, j]


In [ ]:
import numpy as np
from scipy.ndimage import distance_transform_edt

def alpha_from_valid(mask):
    return distance_transform_edt(mask) / (distance_transform_edt(~mask) + 1e-6)

valid1 = (im1_warped.sum(axis=2) > 0)
valid2 = (mosaic.sum(axis=2) > 0)

a1 = alpha_from_valid(valid1)
a2 = alpha_from_valid(valid2)

bias = 1.5
a2 *= bias

W = (a1 + a2 + 1e-6)[..., None]
mosaic_blend = (im1_warped * a1[...,None] + mosaic * a2[...,None]) / W
mosaic_blend = np.clip(mosaic_blend, 0, 255).astype(np.uint8)

In [ ]:
plt.imshow(mosaic_blend.astype(np.uint8))
plt.title("Blended Mosaic")
plt.axis('off')
plt.show()

In [44]:
img = Image.fromarray(mosaic_blend.astype(np.uint8))
img.save("./media/blended_mosaic1.png")